# Autoresearch 3D Post-Hoc Refinement Analysis

Analysis of autonomous refinement experiments logged in `results.tsv` across Track A (`none`), Track B (`oracle`), Track C (`predicted`), and the Déjà View $K$-scaling Pareto curve.

In [ ]:
import json, os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("results.tsv", sep="\t")
for c in ["val_score", "cd_cm", "f1_2cm", "nae_deg", "edge_err_cm", "scene_sec", "memory_gb"]:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")
df["status"] = df["status"].str.strip().str.upper()
df["track"] = df["track"].str.strip().str.lower()

print(f"Total experiments: {len(df)}")
print(df.groupby(["track", "status"]).size().unstack(fill_value=0))
df.head(10)

In [ ]:
track_colors = {"none": "#3498db", "oracle": "#9b59b6", "predicted": "#2ecc71"}
track_labels = {"none": "Track A: No Semantics", "oracle": "Track B: Oracle Semantics", "predicted": "Track C: Predicted Semantics"}

valid = df[df["status"] != "CRASH"].reset_index(drop=True)
fig, ax = plt.subplots(figsize=(15, 7))
ax.axhline(1.0, color="#7f8c8d", linestyle="--", linewidth=1.2, label="Raw Déjà View (K=8) Baseline")

for track, col in track_colors.items():
    sub = valid[valid["track"] == track]
    if sub.empty:
        continue
    disc = sub[sub["status"] == "DISCARD"]
    kept = sub[sub["status"] == "KEEP"]
    ax.scatter(disc.index, disc["val_score"], c=col, s=16, alpha=0.25, zorder=2)
    ax.scatter(kept.index, kept["val_score"], c=col, s=55, edgecolors="black", linewidths=0.6, zorder=4, label=track_labels[track])
    if not kept.empty:
        ax.step(kept.index, kept["val_score"].cummin(), where="post", color=col, linewidth=2.2, alpha=0.85, zorder=3)
        for idx, row in kept.iterrows():
            ax.annotate(str(row["description"]).strip()[:36], (idx, row["val_score"]), textcoords="offset points", xytext=(5, 5), fontsize=7.5, rotation=25, alpha=0.85)

ax.set_xlabel("Experiment #", fontsize=12)
ax.set_ylabel("Composite val_score (lower is better; raw DVLT = 1.0)", fontsize=12)
ax.set_title("Autoresearch 3D Post-Hoc Refinement Progress", fontsize=14)
ax.grid(True, alpha=0.25)
ax.legend(loc="upper right", fontsize=9)
plt.tight_layout()
plt.savefig("progress.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
kept = df[df["status"] == "KEEP"]
summary = (
    kept.sort_values("val_score")
    .groupby("track", as_index=False)
    .first()[["track", "commit", "val_score", "cd_cm", "f1_2cm", "nae_deg", "edge_err_cm", "scene_sec", "description"]]
)
display(summary)

cache_dir = Path(os.environ.get("AUTORESEARCH_3D_CACHE", os.path.expanduser("~/.cache/autoresearch-3d")))
b8_path = cache_dir / "baseline_dev_k8.json"
if b8_path.exists():
    b8 = json.loads(b8_path.read_text())["aggregate"]
    pts = []
    for k in (4, 8, 12, 16):
        p = cache_dir / f"baseline_dev_k{k}.json"
        if p.exists():
            a = json.loads(p.read_text())["aggregate"]
            s = 0.30*(a["cd_cm"]/b8["cd_cm"]) + 0.25*((1-a["f1_2cm"])/(1-b8["f1_2cm"])) + 0.25*(a["nae_deg"]/b8["nae_deg"]) + 0.20*(a["edge_err_cm"]/b8["edge_err_cm"])
            pts.append((k, a["dvlt_runtime_sec"], s))

    fig, ax = plt.subplots(figsize=(10, 6))
    if pts:
        ks, ts, scs = zip(*pts)
        ax.plot(ts, scs, "o-", color="#34495e", linewidth=2, markersize=8, label="Raw Déjà View (K scaling)")
        for k, t, s in pts:
            ax.annotate(f"K={k}", (t, s), textcoords="offset points", xytext=(6, 4), fontsize=9)
    for _, row in summary.iterrows():
        ax.scatter([b8["dvlt_runtime_sec"] + row["scene_sec"]], [row["val_score"]], c=track_colors.get(row["track"], "black"), s=110, edgecolors="black", zorder=5, label=f"DVLT K=8 + {row['track']}")
    ax.set_xlabel("Compute Time per Scene (s)", fontsize=11)
    ax.set_ylabel("Composite val_score (lower is better)", fontsize=11)
    ax.set_title("Quality vs. Compute Pareto Frontier (Déjà View K Scaling vs. Post-Hoc Refinement)", fontsize=13)
    ax.grid(True, alpha=0.25)
    ax.legend(fontsize=9)
    plt.tight_layout()
    plt.savefig("pareto.png", dpi=150, bbox_inches="tight")
    plt.show()